# Notebook 03 – Mô hình và xử lý dữ liệu mất cân bằng

So sánh **Logistic Regression (WoE)**, **Random Forest**, **LightGBM** với các chiến lược xử lý mất cân bằng: không xử lý, trọng số lớp, undersampling, SMOTE.

Nguyên tắc chống rò rỉ:
- Chia train/valid/test **theo thời gian** (`WEEK_NUM`), test là các tuần cuối.
- Mọi bước có fit (WoE, scaler, SMOTE) nằm trong pipeline, chỉ học từ tập train.
- SMOTE chỉ là kỹ thuật huấn luyện trên tập train; valid/test giữ nguyên dữ liệu thật.
- Chọn mô hình theo tập valid; tập test chỉ dùng để báo cáo.

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

import joblib, lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
SEED = 42

## 1. Dữ liệu và chia tập

In [ ]:
df = pl.read_parquet(OUT_DIR / "features_v1.parquet").to_pandas()
feats = [c for c in df.columns if c not in META]
train, valid, test = utils.time_split(df)
for n, d in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{n}: {len(d):>9,} ho so | tuan {d.WEEK_NUM.min()}-{d.WEEK_NUM.max()} | vo no {100*d.target.mean():.2f}%")

iv = pd.read_csv(OUT_DIR / "iv_table.csv", index_col=0)["IV"]
TOP_K = 60                                        # so bien cho LR / RF
top_feats = [c for c in iv.index if c in feats][:TOP_K]
N_SUB = 300_000                                   # mau train cho LR / RF (LightGBM dung toan bo)
tr_sub = train.sample(min(len(train), N_SUB), random_state=SEED)

In [ ]:
results, preds = [], {}

def run(name, model, X_tr, y_tr, cols, fit_kw=None):
    model.fit(X_tr[cols], y_tr, **(fit_kw or {}))
    row = {"mo_hinh": name}
    for split, d in [("train", X_tr), ("valid", valid), ("test", test)]:
        p = model.predict_proba(d[cols])[:, 1]
        m = utils.evaluate(d["target"], p)
        row.update({f"{k}_{split}": v for k, v in m.items() if k in ("AUC", "PR_AUC", "KS", "Brier")})
        if split != "train": preds[(name, split)] = p
    results.append(row); print(name, "| AUC valid = %.4f" % row["AUC_valid"])
    return model

## 2. Logistic Regression + WoE (mô hình nền)

In [ ]:
def make_lr(sampler=None, class_weight=None):
    steps = [("woe", utils.WoEEncoder()), ("scale", StandardScaler())]
    if sampler is not None: steps.append(("sampler", sampler))
    steps.append(("clf", LogisticRegression(max_iter=1000, C=0.1, class_weight=class_weight)))
    return Pipeline(steps)

lr_models = {
    "LR | khong xu ly":    make_lr(),
    "LR | class_weight":   make_lr(class_weight="balanced"),
    "LR | undersampling":  make_lr(RandomUnderSampler(sampling_strategy=0.2, random_state=SEED)),
    "LR | SMOTE":          make_lr(SMOTE(sampling_strategy=0.2, random_state=SEED)),
}
for name, m in lr_models.items():
    run(name, m, tr_sub, tr_sub["target"], top_feats)

## 3. Random Forest

In [ ]:
def make_rf(class_weight=None):
    return Pipeline([("woe", utils.WoEEncoder()),
                     ("clf", RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=100,
                                                    max_features="sqrt", class_weight=class_weight,
                                                    n_jobs=-1, random_state=SEED))])

run("RF | khong xu ly", make_rf(), tr_sub, tr_sub["target"], top_feats)
run("RF | class_weight", make_rf("balanced_subsample"), tr_sub, tr_sub["target"], top_feats);

## 4. LightGBM (toàn bộ đặc trưng, xử lý thiếu và biến phân loại trực tiếp)

In [ ]:
LGB_PARAMS = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, max_depth=-1,
                  min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                  reg_lambda=5.0, max_bin=255, random_state=SEED, n_jobs=-1, verbose=-1,
                  metric="auc")   # chi theo doi AUC; mac dinh LightGBM con theo doi them binary_logloss
fit_kw = dict(eval_set=[(valid[feats], valid["target"])], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
spw = (train["target"] == 0).sum() / (train["target"] == 1).sum()
print("scale_pos_weight day du = %.1f | can bac hai = %.1f" % (spw, np.sqrt(spw)))

lgb_plain = run("LGBM | khong xu ly", lgb.LGBMClassifier(**LGB_PARAMS), train, train["target"], feats, fit_kw)
lgb_spw = run("LGBM | scale_pos_weight", lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=spw), train, train["target"], feats, fit_kw)
lgb_spw_sqrt = run("LGBM | sqrt(scale_pos_weight)", lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=np.sqrt(spw)), train, train["target"], feats, fit_kw)

# So cay thuc su duoc dung sau early stopping: so cay qua it -> mo hinh dung qua som
for n, m in [("khong xu ly", lgb_plain), ("scale_pos_weight", lgb_spw), ("sqrt(scale_pos_weight)", lgb_spw_sqrt)]:
    print(f"{n:<24} best_iteration = {m.best_iteration_}")

**Kiểm tra `scale_pos_weight`.** Lần chạy trước, trọng số đầy đủ (≈ 29) chỉ cho AUC valid 0,72 so với 0,83 của bản không xử lý.
Nguyên nhân nằm ở cấu hình dừng sớm: LightGBM mặc định theo dõi **cả** `binary_logloss` lẫn `auc` trên tập valid và dừng khi *một trong hai* không cải thiện. Với trọng số lớn, xác suất dự đoán bị đẩy lên cao nên logloss trên valid xấu đi gần như ngay từ đầu → mô hình dừng sau rất ít cây, trước khi AUC kịp tăng.
Bản sửa: đặt `metric="auc"` để chỉ dừng theo AUC, nới `early_stopping` lên 200 vòng, in `best_iteration` của từng biến thể và thêm biến thể **căn bậc hai trọng số** (≈ 5,4).
Cách đọc: nếu sau khi sửa ba biến thể có AUC gần nhau thì chênh lệch cũ là do cấu hình chứ không phải do trọng số; khi đó vẫn chọn mô hình theo AUC valid, còn mục tiêu tăng Recall được xử lý bằng **chọn ngưỡng** ở notebook 04 (AUC chỉ phụ thuộc thứ hạng). Brier của bản có trọng số sẽ xấu hơn vì xác suất bị lệch – đã được hiệu chỉnh isotonic ở notebook 04.

### (Tuỳ chọn) Tinh chỉnh bằng Optuna
Đặt `RUN_OPTUNA = True` khi đã có kết quả nền; tinh chỉnh trên mẫu train để tiết kiệm thời gian.

In [ ]:
RUN_OPTUNA, N_TRIALS = False, 30
if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(t):
        p = dict(LGB_PARAMS, num_leaves=t.suggest_int("num_leaves", 15, 127),
                 min_child_samples=t.suggest_int("min_child_samples", 50, 500),
                 colsample_bytree=t.suggest_float("colsample_bytree", 0.3, 0.9),
                 reg_lambda=t.suggest_float("reg_lambda", 0.1, 20, log=True), learning_rate=0.05)
        m = lgb.LGBMClassifier(**p).fit(tr_sub[feats], tr_sub["target"], **fit_kw)
        return utils.evaluate(valid["target"], m.predict_proba(valid[feats])[:, 1])["AUC"]
    study = optuna.create_study(direction="maximize"); study.optimize(objective, n_trials=N_TRIALS)
    print(study.best_value, study.best_params)
    lgb_tuned = run("LGBM | Optuna", lgb.LGBMClassifier(**dict(LGB_PARAMS, **study.best_params)),
                    train, train["target"], feats, fit_kw)

## 5. So sánh kết quả

In [ ]:
res = pd.DataFrame(results).set_index("mo_hinh")
res["chenh_AUC_train_test"] = res["AUC_train"] - res["AUC_test"]
res.to_csv(OUT_DIR / "model_comparison.csv")
res[["AUC_train", "AUC_valid", "AUC_test", "PR_AUC_test", "KS_test", "Brier_test", "chenh_AUC_train_test"]].round(4)

**Cách đọc bảng**
- Xử lý mất cân bằng thường **không** làm tăng AUC nhiều; tác dụng chính là thay đổi Recall ở một ngưỡng và làm **lệch xác suất** (Brier xấu đi) → cần hiệu chỉnh ở notebook 04.
- `chenh_AUC_train_test` > 0,05: dấu hiệu overfit. AUC test > 0,9: nghi ngờ rò rỉ dữ liệu.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["LR | class_weight", "RF | class_weight", "LGBM | khong xu ly", "LGBM | scale_pos_weight", "LGBM | sqrt(scale_pos_weight)"]:
    p = preds[(name, "test")]
    fpr, tpr, _ = roc_curve(test["target"], p); ax[0].plot(fpr, tpr, label=name)
    pr, rc, _ = precision_recall_curve(test["target"], p); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title("ROC (test)"); ax[0].legend()
ax[1].set_title("Precision-Recall (test)"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
plt.tight_layout(); plt.show()

## 6. Lưu mô hình tốt nhất (chọn theo AUC valid) và dự đoán cho notebook 04

In [ ]:
lgb_models = {"LGBM | khong xu ly": lgb_plain, "LGBM | scale_pos_weight": lgb_spw,
              "LGBM | sqrt(scale_pos_weight)": lgb_spw_sqrt}
if RUN_OPTUNA: lgb_models["LGBM | Optuna"] = lgb_tuned
best_name = res.loc[list(lgb_models), "AUC_valid"].idxmax()
best = lgb_models[best_name]; print("Mo hinh chon:", best_name)

joblib.dump({"model": best, "features": feats, "name": best_name}, OUT_DIR / "best_model.joblib")
for split, d in [("valid", valid), ("test", test)]:
    out = d[["case_id", "WEEK_NUM", "target"]].copy(); out["pd_raw"] = preds[(best_name, split)]
    out.to_parquet(OUT_DIR / f"preds_{split}.parquet", index=False)
# Luu du doan cua moi mo hinh (dung cho kiem dinh DeLong o notebook 06)
for split, d in [("valid", valid), ("test", test)]:
    allp = d[["case_id", "WEEK_NUM", "target"]].copy()
    for (name, sp), p in preds.items():
        if sp == split: allp[name] = p
    allp.to_parquet(OUT_DIR / f"preds_all_{split}.parquet", index=False)
train[["case_id"]].assign(pd_raw=best.predict_proba(train[feats])[:, 1]).to_parquet(OUT_DIR / "preds_train.parquet", index=False)

imp = pd.Series(best.booster_.feature_importance("gain"), index=feats).sort_values(ascending=False)
(imp / imp.sum()).head(20).iloc[::-1].plot.barh(figsize=(7, 6), title="Top 20 dac trung (gain)"); plt.show()
print("Ty trong dac trung lon nhat: %.1f%% (qua lon -> kiem tra ro ri)" % (100 * imp.iloc[0] / imp.sum()))